In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Try to get the dataframe from memory or load it
if 'processed_all_train_data' in locals():
    df_check = processed_all_train_data
elif os.path.exists('./data/train_data.csv'):
    print("Loading data from disk...")
    df_check = pl.read_csv('./data/train_data.csv')
else:
    df_check = None
    print("Data not found. Please ensure the training data processing cells have been run.")

if df_check is not None:
    # Display summary statistics
    print("Target (Average Delta Run Exp) Statistics:")
    print(df_check['target'].describe())

    # Plot distribution
    plt.figure(figsize=(10, 6))
    # Sample data for plotting if it's too large
    plot_data = df_check['target'].sample(n=100000, seed=42) if df_check.height > 100000 else df_check['target']

    sns.histplot(plot_data.to_numpy(), bins=50, kde=True)
    plt.title('Distribution of Target Values (Avg Delta Run Exp)')
    plt.xlabel('Target Value (Runs)')
    plt.ylabel('Frequency')
    plt.grid(alpha=0.3)
    plt.show()

    # Display some examples of high/low values
    print("\nExamples of distinct target values:")
    print(df_check.select(['des_new', 'count', 'target']).unique().sort('target').head(5))
    print(df_check.select(['des_new', 'count', 'target']).unique().sort('target', descending=True).head(5))

In [ ]:
!cp -r "/content/drive/MyDrive/stuff_model/data/" "/content/"

In [ ]:
import polars as pl
import numpy as np
import pandas as pd

In [ ]:
# Define a dictionary to group outcomes together
des_dict = {
    'ball': 'ball',
    'hit_into_play': 'hit_into_play',
    'called_strike': 'called_strike',
    'foul': 'foul',
    'swinging_strike': 'swinging_strike',
    'blocked_ball': 'ball',
    'swinging_strike_blocked': 'swinging_strike',
    'foul_tip': 'swinging_strike',
    'foul_bunt': 'foul',
    'hit_by_pitch': 'hit_by_pitch',
    'pitchout': 'ball',
    'missed_bunt': 'swinging_strike',
    'bunt_foul_tip': 'swinging_strike',
    'foul_pitchout': 'foul',
}

# Define a dictionary to group events together
ev_dict = {
    'single': 'single',
    'walk': 'walk',
    'strikeout': 'strikeout',
    'field_out': 'field_out',
    'force_out': 'field_out',
    'double': 'double',
    'hit_by_pitch': 'hit_by_pitch',
    'home_run': 'home_run',
    'grounded_into_double_play': 'field_out',
    'fielders_choice_out': 'field_out',
    'fielders_choice': 'field_out',
    'field_error': 'field_out',
    'double_play': 'field_out',
    'sac_fly': 'field_out',
    'triple': 'triple',
    'sac_bunt': 'field_out',
    'wild_pitch': 'wild_pitch',
    'sac_fly_double_play': 'field_out',
    'triple_play': 'field_out',
    'other_out': 'field_out',
    'sac_bunt_double_play': 'field_out',
}

In [ ]:
swing_descriptions = ['foul_bunt', 'foul', 'hit_into_play', 'swinging_strike', 'foul_tip', 'swinging_strike_blocked', 'missed_bunt', 'bunt_foul_tip', 'foul_pitchout']

In [ ]:
def df_clean(df):
    """
    Clean and transform baseball data using Polars.
    NOTE: Does NOT calculate target values - those must be calculated separately
    from training data only to avoid data leakage.

    Args:
        df: Polars DataFrame with baseball pitch data

    Returns:
        Cleaned and transformed Polars DataFrame (without target column)
    """

    # Create new column with grouped descriptions
    df = df.with_columns([
        pl.col("description").replace_strict(des_dict, default = None).alias("des_new")
    ])

    # Create new column with grouped events for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("events").replace_strict(ev_dict, default = None))
        .otherwise(None)
        .alias("ev_new")
    ])

    # Replace des_new with ev_new for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("ev_new"))
        .otherwise(pl.col("des_new"))
        .alias("des_new")
    ]).drop("ev_new")

    # Filter out null des_new values
    df = df.filter(pl.col("des_new").is_not_null())

    # Filter rare cases (strikes <= 2 and balls <= 3)
    df = df.filter(
        (pl.col("strikes") <= 2) &
        (pl.col("balls") <= 3)
    )

    # Create count column
    df = df.with_columns([
        (pl.col("balls").cast(pl.Utf8) + "-" + pl.col("strikes").cast(pl.Utf8)).alias("count")
    ])

    # Add binary column for swings
    df = df.with_columns([
        pl.col("description").is_in(swing_descriptions).alias("swing")
    ])

    # Cast the year column to integer type
    df = df.with_columns(
        pl.col('game_year').cast(pl.Int64)
    )

    return df

In [ ]:
def calculate_target_lookup(train_df):
    """
    Calculate average run expectancy values from TRAINING data only.
    These values will be applied to both train and test sets.

    Args:
        train_df: Cleaned training DataFrame (output of df_clean)

    Returns:
        DataFrame with average delta_run_exp by des_new, count, p_throws, stand
    """
    target_lookup = train_df.group_by(["des_new", "count", "p_throws", "stand"]).agg([
        pl.col("delta_run_exp").mean().alias("target")
    ])

    return target_lookup


def apply_target_values(df, target_lookup):
    """
    Apply pre-calculated target values to a DataFrame.

    Args:
        df: Cleaned DataFrame (train or test)
        target_lookup: Target lookup table from calculate_target_lookup

    Returns:
        DataFrame with target column added
    """
    # Convert to categorical for efficient joining
    df = df.with_columns([
        pl.col("count").cast(pl.Categorical),
        pl.col("p_throws").cast(pl.Categorical),
        pl.col("stand").cast(pl.Categorical)
    ])

    # Join the target values
    df = df.join(
        target_lookup,
        on = ["des_new", "count", "p_throws", "stand"],
        how = "left"
    )

    return df


def calculate_pitcher_stats_lookup(train_df):
    """
    Calculate pitcher statistics from TRAINING data only.

    Args:
        train_df: Cleaned training DataFrame (output of df_clean)

    Returns:
        DataFrame with pitcher stats (avg_fastball_speed, avg_fastball_az, avg_fastball_ax)
    """
    # Define the pitch types to be considered
    fastball_pitch_types = ['SI', 'FF', 'FC']

    df_filtered = train_df.filter(pl.col('pitch_type').is_in(fastball_pitch_types))

    # Group by pitcher and year, calculate average fastball metrics
    df_agg = df_filtered.group_by(['pitcher', 'game_year', 'pitch_type']).agg([
        pl.col('release_speed').mean().alias('avg_fastball_speed'),
        pl.col('az').mean().alias('avg_fastball_az'),
        pl.col('ax').mean().alias('avg_fastball_ax'),
        pl.len().alias('count')
    ])

    # Sort and get the most-used fastball type
    df_agg = df_agg.sort(['count', 'avg_fastball_speed'], descending = [True, True])
    fastball_stats = df_agg.unique(subset = ['pitcher', 'game_year'])

    # For pitchers without fastballs, find their fastest pitch type
    fastest_pitch_info = (
        train_df.group_by('pitcher')
        .agg([pl.col('release_speed').max().alias('max_speed')])
        .join(train_df, on = 'pitcher')
        .filter(pl.col('release_speed') == pl.col('max_speed'))
        .group_by('pitcher')
        .agg([pl.col('pitch_type').first().alias('fastest_pitch_type')])
    )

    # Calculate stats for fastest pitch (for pitchers without fastballs)
    fastest_pitch_stats = (
        train_df.join(fastest_pitch_info, on = 'pitcher')
        .filter(pl.col('pitch_type') == pl.col('fastest_pitch_type'))
        .group_by(['pitcher', 'game_year'])
        .agg([
            pl.col('release_speed').mean().alias('avg_fastest_pitch_speed'),
            pl.col('az').mean().alias('avg_fastest_pitch_az'),
            pl.col('ax').mean().alias('avg_fastest_pitch_ax')
        ])
    )

    # Combine fastball stats with fastest pitch stats as fallback
    pitcher_stats = (
        fastest_pitch_stats
        .join(fastball_stats.select(['pitcher', 'game_year', 'avg_fastball_speed', 'avg_fastball_az', 'avg_fastball_ax']),
              on = ['pitcher', 'game_year'],
              how = 'left')
        .with_columns([
            pl.coalesce(['avg_fastball_speed', 'avg_fastest_pitch_speed']).alias('avg_fastball_speed'),
            pl.coalesce(['avg_fastball_az', 'avg_fastest_pitch_az']).alias('avg_fastball_az'),
            pl.coalesce(['avg_fastball_ax', 'avg_fastest_pitch_ax']).alias('avg_fastball_ax')
        ])
        .select(['pitcher', 'game_year', 'avg_fastball_speed', 'avg_fastball_az', 'avg_fastball_ax'])
    )

    return pitcher_stats

In [ ]:
def feature_engineering(df: pl.DataFrame, pitcher_stats_lookup: pl.DataFrame) -> pl.DataFrame:
    """
    Apply feature engineering to the dataset using pre-calculated pitcher statistics.

    Args:
        df: Cleaned DataFrame (output of df_clean)
        pitcher_stats_lookup: Pre-calculated pitcher stats from calculate_pitcher_stats_lookup

    Returns:
        DataFrame with engineered features
    """
    # Mirror horizontal break for left-handed pitchers
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('ax'))
        .otherwise(pl.col('ax'))
        .alias('ax')
    )

    # Mirror horizontal release point for left-handed pitchers
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('release_pos_x'))
        .otherwise(pl.col('release_pos_x'))
        .alias('release_pos_x')
    )

    # Join pre-calculated pitcher statistics
    df = df.join(pitcher_stats_lookup, on = ['pitcher', 'game_year'], how = 'left')

    # Calculate pitch differentials
    df = df.with_columns([
        (pl.col('release_speed') - pl.col('avg_fastball_speed')).alias('speed_diff'),
        (pl.col('az') - pl.col('avg_fastball_az')).alias('az_diff'),
        (pl.col('ax') - pl.col('avg_fastball_ax')).alias('ax_diff')
    ])

    return df

In [ ]:
def validate_data(df, dataset_name="dataset"):
    """
    Validate that the dataset has expected structure and values.

    Args:
        df: DataFrame to validate
        dataset_name: Name of dataset for error messages

    Raises:
        ValueError: If validation fails
    """
    required_cols = ['description', 'events', 'strikes', 'balls', 'game_year',
                     'p_throws', 'stand', 'delta_run_exp', 'pitch_type',
                     'ax', 'az', 'release_speed', 'release_pos_x', 'pitcher']

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{dataset_name}: Missing required columns: {missing_cols}")

    # Check for valid strike/ball counts
    invalid_strikes = df.filter((pl.col('strikes') < 0) | (pl.col('strikes') > 3)).height
    invalid_balls = df.filter((pl.col('balls') < 0) | (pl.col('balls') > 4)).height

    if invalid_strikes > 0:
        print(f"Warning: {dataset_name} has {invalid_strikes} rows with invalid strike counts")
    if invalid_balls > 0:
        print(f"Warning: {dataset_name} has {invalid_balls} rows with invalid ball counts")

    # Report null counts in critical columns
    for col in ['pitch_type', 'release_speed', 'ax', 'az', 'delta_run_exp']:
        null_count = df.filter(pl.col(col).is_null()).height
        if null_count > 0:
            print(f"Info: {dataset_name} has {null_count} null values in '{col}' ({null_count/df.height*100:.2f}%)")

    print(f"{dataset_name} validation complete. Shape: {df.shape}")
    return True

In [ ]:
# Load data using Polars
data_2021 = pl.read_csv('./data/2021_data.csv')
data_2022 = pl.read_csv('./data/2022_data.csv')
data_2023 = pl.read_csv('./data/2023_data.csv')
data_2024 = pl.read_csv('./data/2024_data.csv')

In [ ]:
# Concatenate all training years
all_train_data = pl.concat([data_2021, data_2022, data_2023, data_2024], how = 'vertical_relaxed')

# Validate raw data
validate_data(all_train_data, "Raw training data")

Info: Raw training data has 10567 null values in 'pitch_type' (0.37%)
Info: Raw training data has 10347 null values in 'release_speed' (0.36%)
Info: Raw training data has 10351 null values in 'ax' (0.36%)
Info: Raw training data has 10351 null values in 'az' (0.36%)
Info: Raw training data has 9279 null values in 'delta_run_exp' (0.33%)
Raw training data validation complete. Shape: (2854422, 118)


True

In [ ]:
# Step 1: Clean the training data
print("=" * 60)
print("STEP 1: Cleaning training data")
print("=" * 60)
cleaned_train_data = df_clean(all_train_data).filter(pl.col("pitch_type").is_not_null())
print(f"After cleaning: {cleaned_train_data.shape[0]} rows")

# Step 2: Calculate lookup tables from TRAINING data only (prevent data leakage)
print("\n" + "=" * 60)
print("STEP 2: Calculating statistics from training data")
print("=" * 60)
target_lookup = calculate_target_lookup(cleaned_train_data)
print(f"Target lookup table created with {target_lookup.shape[0]} unique combinations")

# FIX: Ensure target_lookup's join key columns are also categorical before applying target values
target_lookup = target_lookup.with_columns(
    pl.col("count").cast(pl.Categorical),
    pl.col("p_throws").cast(pl.Categorical),
    pl.col("stand").cast(pl.Categorical)
)

pitcher_stats_lookup = calculate_pitcher_stats_lookup(cleaned_train_data)
print(f"Pitcher stats lookup created for {pitcher_stats_lookup.select('pitcher').unique().height} pitchers")

# Step 3: Apply target values to training data
print("\n" + "=" * 60)
print("STEP 3: Applying target values to training data")
print("=" * 60)
train_with_target = apply_target_values(cleaned_train_data, target_lookup)
print(f"Target values applied. Rows with null target: {train_with_target.filter(pl.col('target').is_null()).height}")

# Step 4: Apply feature engineering to training data
print("\n" + "=" * 60)
print("STEP 4: Feature engineering on training data")
print("=" * 60)
processed_all_train_data = feature_engineering(train_with_target, pitcher_stats_lookup)
print(f"Feature engineering complete. Final shape: {processed_all_train_data.shape}")

STEP 1: Cleaning training data
After cleaning: 2843813 rows

STEP 2: Calculating statistics from training data
Target lookup table created with 479 unique combinations
Pitcher stats lookup created for 1569 pitchers

STEP 3: Applying target values to training data


/tmp/ipython-input-521746851.py:38: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  df = df.join(


Target values applied. Rows with null target: 0

STEP 4: Feature engineering on training data
Feature engineering complete. Final shape: (2843813, 128)


In [ ]:
# Optional: Save lookup tables for future use with test data
print("\n" + "=" * 60)
print("SAVING LOOKUP TABLES")
print("=" * 60)
target_lookup.write_csv('./data/target_lookup.csv')
print("Saved: target_lookup.csv")

pitcher_stats_lookup.write_csv('./data/pitcher_stats_lookup.csv')
print("Saved: pitcher_stats_lookup.csv")


SAVING LOOKUP TABLES
Saved: target_lookup.csv
Saved: pitcher_stats_lookup.csv


In [ ]:
print("\n" + "=" * 60)
print("SAVING FINAL TRAINING DATA")
print("=" * 60)
processed_all_train_data.write_csv('./data/train_data.csv')
print(f"Saved train_data.csv with {processed_all_train_data.shape[0]} rows and {processed_all_train_data.shape[1]} columns")


SAVING FINAL TRAINING DATA
Saved train_data.csv with 2843813 rows and 128 columns


In [ ]:
!cp "./data/train_data.csv" "/content/drive/MyDrive/stuff_model/data/"

In [ ]:
# ============================================================================
# PROCESS 2025 DATA (NEW PITCHES TO SCORE)
# ============================================================================

# Load 2025 data
print("=" * 60)
print("LOADING 2025 DATA")
print("=" * 60)
data_2025 = pl.read_csv('./data/2025_data.csv')
validate_data(data_2025, "Raw 2025 data")

# Step 1: Clean 2025 data
print("\n" + "=" * 60)
print("STEP 1: Cleaning 2025 data")
print("=" * 60)
cleaned_2025 = df_clean(data_2025).filter(pl.col("pitch_type").is_not_null())
print(f"After cleaning: {cleaned_2025.shape[0]} rows")

# Step 2: Calculate pitcher stats specific to 2025
# We calculate stats from the 2025 data itself so 'speed_diff' is relative to
# the pitcher's current year average, which is standard for Stuff+ models.
print("\n" + "=" * 60)
print("STEP 2: Calculating pitcher stats for 2025")
print("=" * 60)
pitcher_stats_2025 = calculate_pitcher_stats_lookup(cleaned_2025)
print(f"Pitcher stats lookup created for {pitcher_stats_2025.select('pitcher').unique().height} pitchers in 2025")

# Step 3: Apply feature engineering using 2025 stats
print("\n" + "=" * 60)
print("STEP 3: Feature engineering on 2025 data")
print("=" * 60)
# Use the 2025 stats here so the 'game_year' join works correctly
processed_2025 = feature_engineering(cleaned_2025, pitcher_stats_2025)
print(f"Feature engineering complete. Final shape: {processed_2025.shape}")

# Step 4: Save processed 2025 data
print("\n" + "=" * 60)
print("SAVING PROCESSED 2025 DATA")
print("=" * 60)
processed_2025.write_csv('./data/test_data_2025.csv')
print(f"Saved test_data_2025.csv with {processed_2025.shape[0]} rows and {processed_2025.shape[1]} columns")

LOADING 2025 DATA
Info: Raw 2025 data has 12324 null values in 'pitch_type' (1.66%)
Info: Raw 2025 data has 12313 null values in 'release_speed' (1.66%)
Info: Raw 2025 data has 12314 null values in 'ax' (1.66%)
Info: Raw 2025 data has 12314 null values in 'az' (1.66%)
Info: Raw 2025 data has 2505 null values in 'delta_run_exp' (0.34%)
Raw 2025 data validation complete. Shape: (742080, 118)

STEP 1: Cleaning 2025 data
After cleaning: 729740 rows

STEP 2: Calculating pitcher stats for 2025
Pitcher stats lookup created for 1044 pitchers in 2025

STEP 3: Feature engineering on 2025 data
Feature engineering complete. Final shape: (729740, 127)

SAVING PROCESSED 2025 DATA
Saved test_data_2025.csv with 729740 rows and 127 columns


In [ ]:
!cp "./data/test_data_2025.csv" "/content/drive/MyDrive/stuff_model/data/"